<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Exploratory Data Analysis with Sales Data</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Sanchaita Chakraborty<br>
  <strong>Designation:</strong> Project Linked Associate Research Engineer<br>
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Import Data and Store-wise Revenue Analysis (4 Marks)

Import `pandas` and `numpy`. Download the dataset `sales.csv` from https://drive.google.com/drive/folders/1gieHICVDBbUKMZiSF4YRQMAJic-w50JM?usp=sharing. Load the dataset `sales.csv` into a DataFrame. Then, find the top 5 stores with the highest total sales and the bottom 5 stores with the lowest total sales.

**Tasks:**
- Aggregate total sales by store_id
- Rank stores by total revenue
- Compare average sales per department within those stores

In [ ]:
# Write your answer here
import pandas as pd

# Load the dataset
df_sales = pd.read_csv('sales.csv')

# Aggregate total sales per store to establish rankings
store_sales = df_sales.groupby('store_id')['weekly_sales'].sum().reset_index()
store_sales.rename(columns={'weekly_sales': 'total_sales'}, inplace=True)

# Rank stores by total revenue (highest to lowest)
ranked_stores = store_sales.sort_values(by='total_sales', ascending=False).reset_index(drop=True)
ranked_stores['rank'] = ranked_stores.index + 1

# Extract the cohort boundary targets (top 5 and bottom 5 stores)
top_5_stores = ranked_stores.head(5)
bottom_5_stores = ranked_stores.tail(5)

# ==========================================================
# OP 1: Rank stores by total revenue (Top 5 & Bottom 5)
# ==========================================================
print("--- OP 1: Ranked Stores by Total Revenue ---")
print("Top 5 Stores:")
print(top_5_stores.to_string(index=False))
print("\nBottom 5 Stores:")
print(bottom_5_stores.to_string(index=False))
print("\n")

# ==========================================================
# OP 2: Compare average sales per department within cohorts
# ==========================================================
# Build the ordered target store list maintaining revenue rank order
ordered_store_ids = pd.concat([top_5_stores, bottom_5_stores])['store_id'].tolist()

# Filter master transactional data for these specific cohorts
cohort_transactions = df_sales[df_sales['store_id'].isin(ordered_store_ids)]

# Compute chronological mean sales per department
dept_averages = cohort_transactions.groupby(['store_id', 'department'])['weekly_sales'].mean().reset_index()
dept_averages.rename(columns={'weekly_sales': 'average_weekly_sales'}, inplace=True)

# Pivot flat data into a cross-operational matrix
pivoted_comparison = dept_averages.pivot(
    index='department', 
    columns='store_id', 
    values='average_weekly_sales'
).fillna(0.0)

# Explicitly sort columns based on the total revenue rank sequence
pivoted_comparison_ranked = pivoted_comparison.reindex(columns=ordered_store_ids)

# Filter down to a core sample of departments (e.g., 1 to 5) for cleaner reporting
target_departments = [1, 2, 3, 4, 5]
filtered_pivot = pivoted_comparison_ranked.loc[target_departments]

print("--- OP 2: Departmental Average Weekly Sales Comparison (Ranked Columns) ---")
print(filtered_pivot.round(2))

### Question 2: Department Performance Consistency (3 Marks)

Which departments have the most stable weekly sales and which are the most volatile?

**Tasks:**
- Use standard deviation and coefficient of variation
- Identify highly stable and highly volatile departments

In [ ]:
# Write your answer here
import pandas as pd

# Load dataset
df_sales = pd.read_csv('sales.csv')

# Compute baseline metrics per department
dept_stats = df_sales.groupby('department')['weekly_sales'].agg(['mean', 'std']).reset_index()
dept_stats.rename(columns={'std': 'standard_deviation'}, inplace=True)

# Compute Coefficient of Variation (CV)
dept_stats['coefficient_of_variation'] = dept_stats['standard_deviation'] / dept_stats['mean']


# =========================================================================
# LENS 1: Absolute Volatility (Using Standard Deviation Alone)
# =========================================================================
stable_by_sd = dept_stats.sort_values(by='standard_deviation', ascending=True).head(5)
volatile_by_sd = dept_stats.sort_values(by='standard_deviation', ascending=False).head(5)

print("=== METHOD 1: ABSOLUTE VOLATILITY (STANDARD DEVIATION) ===")
print("\nTop 5 Highly Stable Departments (Lowest Raw Dollar Variance):")
print(stable_by_sd[['department', 'standard_deviation', 'mean']].round(2).to_string(index=False))

print("\nTop 5 Highly Volatile Departments (Highest Raw Dollar Variance):")
print(volatile_by_sd[['department', 'standard_deviation', 'mean']].round(2).to_string(index=False))
print("\n" + "="*60 + "\n")


# =========================================================================
# LENS 2: Relative Volatility (Using Coefficient of Variation Alone)
# =========================================================================
stable_by_cv = dept_stats.sort_values(by='coefficient_of_variation', ascending=True).head(5)
volatile_by_cv = dept_stats.sort_values(by='coefficient_of_variation', ascending=False).head(5)

print("=== METHOD 2: RELATIVE VOLATILITY (COEFFICIENT OF VARIATION) ===")
print("\nTop 5 Highly Stable Departments (Lowest Proportional Variance):")
print(stable_by_cv[['department', 'coefficient_of_variation', 'standard_deviation', 'mean']].round({'mean': 2, 'standard_deviation': 2, 'coefficient_of_variation': 4}).to_string(index=False))

print("\nTop 5 Highly Volatile Departments (Highest Proportional Variance):")
print(volatile_by_cv[['department', 'coefficient_of_variation', 'standard_deviation', 'mean']].round({'mean': 2, 'standard_deviation': 2, 'coefficient_of_variation': 4}).to_string(index=False))

### Question 3: Holiday vs Non-Holiday Sales (3 Marks)

Analyze whether holidays significantly affect weekly sales.

**Tasks:**
- Compare average sales during holidays vs non-holidays
- Calculate percentage increase/decrease
- Identify departments benefiting most from holidays

In [ ]:
# Write your answer here
import pandas as pd

# Load dataset
df_sales = pd.read_csv('sales.csv')

# =========================================================================
# Task 1 & 2: Overall Holiday vs. Non-Holiday Impact
# =========================================================================
overall_stats = df_sales.groupby('is_holiday')['weekly_sales'].mean().reset_index()

non_holiday_avg = overall_stats[overall_stats['is_holiday'] == 0]['weekly_sales'].values[0]
holiday_avg = overall_stats[overall_stats['is_holiday'] == 1]['weekly_sales'].values[0]

# Calculate percentage increase
pct_change = ((holiday_avg - non_holiday_avg) / non_holiday_avg) * 100

print("=== OVERALL HOLIDAY IMPACT ON SALES ===")
print(f"Average Weekly Sales (Regular Weeks): ${non_holiday_avg:,.2f}")
print(f"Average Weekly Sales (Holiday Weeks): ${holiday_avg:,.2f}")
print(f"Percentage Revenue Shift on Holidays: +{pct_change:.2f}%\n")


# =========================================================================
# Task 3: Departmental Deep Dive (Who benefits most?)
# =========================================================================
# Unstack regular vs holiday averages by department
dept_holiday_stats = df_sales.groupby(['department', 'is_holiday'])['weekly_sales'].mean().unstack()
dept_holiday_stats.columns = ['regular_weeks_avg', 'holiday_weeks_avg']

# Calculate the exact percentage increase per department
dept_holiday_stats['pct_increase'] = (
    (dept_holiday_stats['holiday_weeks_avg'] - dept_holiday_stats['regular_weeks_avg']) 
    / dept_holiday_stats['regular_weeks_avg']
) * 100

# Extract top performing departments
top_beneficiaries = dept_holiday_stats.sort_values(by='pct_increase', ascending=False).head(5)

print("=== TOP 5 DEPARTMENTS BENEFITING MOST FROM HOLIDAYS ===")
print(top_beneficiaries.round(2))

### Question 4: Seasonal Trend Detection (3 Marks)

Identify whether sales increase or decrease during specific months.

**Tasks:**
- Extract month from the date column
- Compute average monthly sales
- Determine highest and lowest sales months

In [ ]:
# Write your answer here
import pandas as pd

# Load dataset
df_sales = pd.read_csv('sales.csv')

# Ensure the date column is parsed correctly as a datetime object
df_sales['date'] = pd.to_datetime(df_sales['date'])

# Task 1: Extract month number and full month name from the date column
df_sales['month'] = df_sales['date'].dt.month
df_sales['month_name'] = df_sales['date'].dt.strftime('%B')

# Task 2: Compute average weekly sales for each month
# (We also aggregate total sales to provide complete visibility)
monthly_stats = df_sales.groupby(['month', 'month_name'])['weekly_sales'].agg(
    average_sales='mean',
    total_sales='sum'
).reset_index()

# Sort by average sales in descending order to easily evaluate performance shifts
monthly_stats_sorted = monthly_stats.sort_values(by='average_sales', ascending=False)

print("--- MONTHLY SALES TREND ANALYSIS ---")
print(monthly_stats_sorted.round(2).to_string(index=False))

# Task 3: Determine the highest and lowest sales months
highest_month = monthly_stats_sorted.iloc[0]
lowest_month = monthly_stats_sorted.iloc[-1]

print("\n--- SEASONAL EXTREMES ---")
print(f"Highest Sales Month: {highest_month['month_name']} (Average Weekly Sales: ${highest_month['average_sales']:,.2f})")
print(f"Lowest Sales Month:  {lowest_month['month_name']} (Average Weekly Sales: ${lowest_month['average_sales']:,.2f})")

### Question 5: Store and Department Contribution (4 Marks)

Which combinations of store and department contribute the most to overall revenue?

**Tasks:**
- Group using store_id and department
- Find top and worst-performing combinations
- Compute percentage contribution

In [ ]:
# Write your answer here
import pandas as pd

# Load dataset
df_sales = pd.read_csv('sales.csv')

# Step 1: Calculate the company's gross overall revenue across all stores and departments
total_network_revenue = df_sales['weekly_sales'].sum()

# Step 2: Group by store_id and department and sum up total sales
combination_sales = df_sales.groupby(['store_id', 'department'])['weekly_sales'].sum().reset_index()
combination_sales.rename(columns={'weekly_sales': 'total_combination_revenue'}, inplace=True)

# Step 3: Compute percentage contribution to the overall company revenue
combination_sales['percentage_contribution'] = (combination_sales['total_combination_revenue'] / total_network_revenue) * 100

# Rank the combinations from highest to lowest revenue generators
ranked_combinations = combination_sales.sort_values(by='total_combination_revenue', ascending=False).reset_index(drop=True)

# Separate top 5 and bottom 5 performers
top_5_units = ranked_combinations.head(5)
bottom_5_units = ranked_combinations.tail(5)

print(f"Grand Overall Revenue (All Stores & Depts): ${total_network_revenue:,.2f}\n")

print("--- TOP 5 REVENUE CONTRIBUTING COMBINATIONS ---")
print(top_5_units.round({'total_combination_revenue': 2, 'percentage_contribution': 4}).to_string(index=False))

print("\n--- BOTTOM 5 REVENUE CONTRIBUTING COMBINATIONS ---")
print(bottom_5_units.round({'total_combination_revenue': 2, 'percentage_contribution': 4}).to_string(index=False))

### Question 6: Load MNIST Data (3 Marks)

Load the MNIST dataset using `sklearn.datasets.load_digits`. Separate the dataset into features (`X`) and target labels (`y`).
Print the shape of the features and the target arrays.

**Expected Output:** The shape of `X` and `y`.

In [ ]:
# Write your answer here
from sklearn.datasets import load_digits

# Load the digits dataset
digits = load_digits()

# Separate the dataset into features (X) and target labels (y)
X = digits.data
y = digits.target

# Print the shape of the features and target arrays
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

### Question 7: K-Means Clustering (5 Marks)

Import `KMeans` from `sklearn.cluster`. Initialize and fit the K-Means clustering model on the MNIST features (`X`). 
Since we know there are 10 digits (0-9), set the number of clusters to 10. Assign the cluster labels to a variable `kmeans_labels`.

**Expected Output:** A successfully fitted K-Means model and the cluster labels array.

In [ ]:
# Write your answer here
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans

# 1. Load the digits dataset
digits = load_digits()
X = digits.data

# 2. Initialize the K-Means clustering model
# We set n_clusters=10 because there are 10 distinct digits (0-9)
# random_state is set to ensure reproducibility
kmeans = KMeans(n_clusters=10, random_state=42)

# 3. Fit the model on the features (X)
kmeans.fit(X)

# 4. Assign the resulting cluster labels
kmeans_labels = kmeans.labels_

# Print verification output
print("Successfully fitted model:", kmeans)
print("Cluster labels array shape:", kmeans_labels.shape)
print("First 20 cluster labels:\n", kmeans_labels[:20])

### Question 8: F1 Score Evaluation for Clustering (5 Marks)

Evaluate the clustering performance using the F1 score. Since K-Means assigns arbitrary cluster labels, you will first need to map each cluster label to the most frequent true label in that cluster. 
After mapping the labels, calculate and print the macro-averaged F1 score using `sklearn.metrics.f1_score`.

**Expected Output:** The calculated F1 score of the clustering.

In [ ]:
# Write your answer here
import numpy as np
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score
from scipy.stats import mode

# 1. Load the digits dataset and separate features and target labels
digits = load_digits()
X = digits.data
y = digits.target

# 2. Initialize and fit the K-Means clustering model (10 clusters)
kmeans = KMeans(n_clusters=10, random_state=42)
kmeans.fit(X)
kmeans_labels = kmeans.labels_

# 3. Map each cluster label to the most frequent true label in that cluster
mapped_labels = np.zeros_like(kmeans_labels)

for i in range(10):
    # Mask to identify samples belonging to cluster i
    mask = (kmeans_labels == i)
    
    if np.sum(mask) > 0:
        # Find the most frequent true digit label within this cluster partition
        most_common_true_label = mode(y[mask], keepdims=True).mode[0]
        # Assign this true label back to the corresponding indices
        mapped_labels[mask] = most_common_true_label

# 4. Calculate and print the macro-averaged F1 score
f1 = f1_score(y, mapped_labels, average='macro')
print(f"Calculated Macro-Averaged F1 Score: {f1:.4f}")